In [2]:
import pandas as pd   # for handling tabular data
import numpy as np    # for fast numerical operations

In [3]:
ge1 = pd.read_excel("../data/raw/GSE33000_raw_data_2 class_PFC_dataset_ClassDenoted (2).xlsx")

# Load the gene expression dataset into a pandas DataFrame
# Excel is used because data is stored in .xlsx format

In [4]:
ge1 = ge1.iloc[:, :-1]

# Last column contains an invalid value (157) only in label row
# It is not a real sample → must be removed

In [6]:
ge1_labels = ge1.iloc[-1, 1:].values.astype(int)
ge1_data = ge1.iloc[:-1].copy()

# Last row = class labels (0/1)
# Remaining rows = gene expression data
# .copy() avoids pandas warnings

In [9]:
# Check last 5 column names
print(ge1.columns[-5:])

# Check last row (labels) for last 5 columns
print(ge1.iloc[-1, -5:])

# Check unique values in last row
print(pd.unique(ge1.iloc[-1]))

Index(['PFC_463', 'PFC_464', 'PFC_465', 'PFC_466', 'PFC_467'], dtype='object')
PFC_463    0.0
PFC_464    0.0
PFC_465    0.0
PFC_466    0.0
PFC_467    0.0
Name: 39280, dtype: object
['Class Lebel' np.float64(1.0) np.float64(0.0)]


In [10]:
ge1_data['Gene'] = ge1_data['Gene'].fillna('Unknown')

# WHY:
# Some gene names are missing (NaN)
# Replace with 'Unknown' so indexing does not break later

In [11]:
print(ge1_data['Gene'].isnull().sum())

0


In [12]:
ge1_data.set_index('Gene', inplace=True)

# Gene names should act as feature identifiers (column labels after transpose)
# This removes the text column and keeps only numeric data

In [13]:
print(ge1_data.index[:5])   # check first few gene names
print(ge1_data.shape)      # check shape

Index(['RSE_00000118999', 'AGRN,FLJ45064', 'HMGB2,HMG2', 'LOC285706',
       'DNAH12,DHC3,HL19,DLP12,HL-19,hdhc3,DNAHC3,DNAHC12'],
      dtype='object', name='Gene')
(39280, 467)


In [14]:
ge1_data = pd.DataFrame(ge1_data.values.T, columns=ge1_data.index)

# Current: Genes × Samples  → (39280, 467)
# Needed:  Samples × Genes  → (467, 39280)
# ML models require rows = samples, columns = features
# Using NumPy transpose is faster than pandas .T for large data

In [15]:
print(ge1_data.shape)

(467, 39280)


In [16]:
ge1_data = ge1_data.astype(np.float32)

# Ensure all values are numeric
# float32 reduces memory usage (important for large dataset)
# Prevents computation issues later

In [17]:
print(ge1_data.dtypes.unique())

[dtype('float32')]


In [18]:
data_np = ge1_data.values

# NumPy arrays are much faster than pandas for large computations
# Needed for efficient mean imputation

In [19]:
print(type(data_np))
print(data_np.shape)

<class 'numpy.ndarray'>
(467, 39280)


In [20]:
col_mean = np.nanmean(data_np, axis=0)

# Compute mean for each gene (column)
# Required to replace missing values (mean imputation)
# np.nanmean ignores NaN values automatically

In [21]:
print(col_mean.shape)
print(np.isnan(col_mean).sum())

(39280,)
0


In [22]:
inds = np.where(np.isnan(data_np))

# Find positions (row, column) where values are missing (NaN)
# We will replace these using column mean

In [23]:
print(len(inds[0]))

250991


In [24]:
data_np[inds] = np.take(col_mean, inds[1])

# inds[0] → row indices (samples)
# inds[1] → column indices (genes)
# np.take(col_mean, inds[1]) → picks correct mean for each gene
# This replaces each NaN with its column (gene) mean

In [25]:
print(np.isnan(data_np).sum())

0


In [26]:
ge1_data = pd.DataFrame(data_np, columns=ge1_data.columns)

# WHY:
# We used NumPy for fast computation
# Now convert back to pandas for easier handling
# Keep original gene names as column names

In [27]:
print(type(ge1_data))
print(ge1_data.shape)

<class 'pandas.core.frame.DataFrame'>
(467, 39280)


In [29]:
ge1_data.columns = ge1_data.columns.astype(str)

# Ensure all feature names are strings
# Required for sklearn compatibility

In [30]:
print(type(ge1_data.columns[0]))

<class 'str'>


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
ge1_data = scaler.fit_transform(ge1_data)

# Apply Z-score normalization
# Mean = 0, Standard deviation = 1
# Ensures all genes are on same scale
# Improves ML model performance

In [32]:
print(type(ge1_data))
print(ge1_data.shape)
print(round(ge1_data.mean(), 2))
print(round(ge1_data.std(), 2))

<class 'numpy.ndarray'>
(467, 39280)
0.0
1.0


In [33]:
np.save("../data/processed/GSE33000_data.npy", ge1_data)
np.save("../data/processed/GSE33000_labels.npy", ge1_labels)

# Clear naming → easy to identify dataset later
# Important when working with multiple datasets (GE + DM)

In [34]:
import os
print(os.listdir("../data/processed"))

['.gitkeep', 'GSE33000_data.npy', 'GSE33000_labels.npy']


In [2]:
import pandas as pd   # for handling tabular data
import numpy as np    # for fast numerical operationsge1_data_loaded = np.load("../data/processed/GSE33000_data.npy")
ge1_labels_loaded = np.load("../data/processed/GSE33000_labels.npy")
# Load saved data to verify correctness

In [3]:
print(ge1_data_loaded.shape)
print(len(ge1_labels_loaded))

NameError: name 'ge1_data_loaded' is not defined

In [4]:
ge1_data_loaded = np.load("../data/processed/GSE33000_data.npy")
ge1_labels_loaded = np.load("../data/processed/GSE33000_labels.npy")

# WHY:
# Load saved data to verify correctness

In [5]:
print(ge1_data_loaded.shape)
print(len(ge1_labels_loaded))

(467, 39280)
467


In [6]:
print(ge1_data_loaded[:2, :5])

[[-0.6048827  -0.3641479   1.384371   -0.12960087 -0.78888756]
 [-0.80934465 -0.32800514  0.1976333   0.08960817 -0.68124485]]


In [7]:
print(ge1_data_loaded[:-1, :-5])

[[-6.04882717e-01 -3.64147902e-01  1.38437104e+00 ... -8.68194222e-01
   2.15833947e-01  1.23840861e-01]
 [-8.09344649e-01 -3.28005135e-01  1.97633296e-01 ...  5.05766707e-07
   1.71387509e-01 -1.79536045e-01]
 [-9.62044001e-01 -3.64147902e-01 -4.43229042e-02 ... -8.57261539e-01
  -8.50881755e-01 -9.51251268e-01]
 ...
 [-5.89353919e-01 -3.36036861e-01 -9.66060877e-01 ... -4.38173294e-01
  -2.28631020e-01 -4.44990903e-01]
 [-7.34288990e-01 -4.20369953e-01 -3.09322596e-01 ... -9.59300339e-01
  -6.95318937e-01 -2.93302387e-01]
 [-7.16172099e-01 -2.63751447e-01 -6.20409131e-01 ... -3.10624599e-01
   3.04726809e-01 -1.22652896e-01]]


In [8]:
print(np.mean(ge1_data_loaded))

1.7804371e-08


In [9]:
print(np.std(ge1_data_loaded))

1.0000004


In [10]:
print(np.min(ge1_data_loaded), np.max(ge1_data_loaded))

-7.2584214 21.585682


In [11]:
print((ge1_data_loaded > 10).sum())

4027


In [13]:
# Preprocessing: GSE44770 (Gene Expression - TXT Dataset)

#This section preprocesses the GSE44770 dataset. The structure is different from GSE33000, so we will first inspect and then process accordingly.

In [17]:
ge2 = pd.read_csv(
    "../data/raw/GSE44770_series_matrix_ClassLabel.txt",
    sep="\t",
    comment="!",
    header=None,
    low_memory=False
)

# header=None → do NOT treat first row as column names
# comment="!" → remove GEO metadata lines
# low_memory=False → avoid dtype issues

In [18]:
print(ge2.shape)
ge2.head()

(39282, 232)


,0,1,2,3,4,5,6,7,8,9,...,222,223,224,225,226,227,228,229,230,231
0,"Class Label (AD-1, N-0)",1,1,0,0,1,1,0,1,0,...,1,0,0,0,0,1,1,1,1,100.0
1,ID_REF,GSM1090501,GSM1090502,GSM1090503,GSM1090504,GSM1090505,GSM1090506,GSM1090507,GSM1090508,GSM1090509,...,GSM1090722,GSM1090723,GSM1090724,GSM1090725,GSM1090726,GSM1090727,GSM1090728,GSM1090729,GSM1090730,NaN
2,10019475365,0.0276,-0.0205,0.112,-0.0816,-0.0517,-0.0466,-0.0314,0.112,0.0283,...,0.0415,0.122,-0.0349,0.115,0.074,0.0766,0.0326,0.228,0.132,NaN
3,10019481149,-0.0236,0.0034,0.0437,-0.0032,-0.0641,-0.0718,-0.0182,-0.0517,0.0175,...,0.0154,0.0106,-0.0101,0.0981,0.0172,0.0164,0.0296,0.0745,0.0628,NaN
4,10019495284,-0.0587,0.0727,-0.141,0.0389,-0.051,-0.075,-0.254,-0.0907,-0.0997,...,0.0336,-0.259,-0.16,-0.191,-0.182,-0.214,-0.0804,-0.19,0.0019,NaN


In [19]:
ge2 = ge2.iloc[:, :-1]

# Last column contains invalid values (100 / NaN)
# Not a real sample → must remove

In [20]:
print(ge2.shape)
ge2.iloc[0, -5:]

(39282, 231)


226    0
227    1
228    1
229    1
230    1
Name: 0, dtype: object

In [21]:
ge2_labels = ge2.iloc[0, 1:].values.astype(int)

# Row 0 → contains class labels
# Skip first column (text: "Class Label")
# Convert to integer (0/1)

In [22]:
print(len(ge2_labels))
print(ge2_labels[:10])
print(set(ge2_labels))

230
[1 1 0 0 1 1 0 1 0 1]
{np.int64(0), np.int64(1)}


In [23]:
ge2_data = ge2.iloc[2:].copy()

# Row 0 → labels (already extracted)
# Row 1 → sample IDs (not needed for ML)
# Remaining rows → actual gene expression data

In [24]:
print(ge2_data.shape)
ge2_data.head()

(39280, 231)


,0,1,2,3,4,5,6,7,8,9,...,221,222,223,224,225,226,227,228,229,230
2,10019475365,0.0276,-0.0205,0.112,-0.0816,-0.0517,-0.0466,-0.0314,0.112,0.0283,...,0.0183,0.0415,0.122,-0.0349,0.115,0.074,0.0766,0.0326,0.228,0.132
3,10019481149,-0.0236,0.0034,0.0437,-0.0032,-0.0641,-0.0718,-0.0182,-0.0517,0.0175,...,0.0143,0.0154,0.0106,-0.0101,0.0981,0.0172,0.0164,0.0296,0.0745,0.0628
4,10019495284,-0.0587,0.0727,-0.141,0.0389,-0.051,-0.075,-0.254,-0.0907,-0.0997,...,-0.173,0.0336,-0.259,-0.16,-0.191,-0.182,-0.214,-0.0804,-0.19,0.0019
5,10019687586,0.0329,-0.0331,0.159,0.0279,-0.0492,0.0725,-0.0281,0.127,0.042,...,0.0051,-0.0118,-0.145,-0.0683,-0.0757,-0.0749,-0.213,-0.101,-0.0948,0.0056
6,10019713746,0.0325,-0.0103,0.164,-0.0884,0.0631,-0.0556,0.0209,0.127,0.0115,...,0.0301,0.027,0.0856,-0.0098,0.0457,0.0579,-0.026,0.178,0.232,0.0497


In [25]:
gene_ids = ge2_data.iloc[:, 0]
ge2_data = ge2_data.iloc[:, 1:]

# WHY:
# First column contains gene identifiers (not numeric features)
# Remaining columns are expression values → actual features for ML

In [26]:
print(gene_ids.head())
print(ge2_data.shape)
ge2_data.head()

2    10019475365
3    10019481149
4    10019495284
5    10019687586
6    10019713746
Name: 0, dtype: object
(39280, 230)


,1,2,3,4,5,6,7,8,9,10,...,221,222,223,224,225,226,227,228,229,230
2,0.0276,-0.0205,0.112,-0.0816,-0.0517,-0.0466,-0.0314,0.112,0.0283,-0.0331,...,0.0183,0.0415,0.122,-0.0349,0.115,0.074,0.0766,0.0326,0.228,0.132
3,-0.0236,0.0034,0.0437,-0.0032,-0.0641,-0.0718,-0.0182,-0.0517,0.0175,-0.0757,...,0.0143,0.0154,0.0106,-0.0101,0.0981,0.0172,0.0164,0.0296,0.0745,0.0628
4,-0.0587,0.0727,-0.141,0.0389,-0.051,-0.075,-0.254,-0.0907,-0.0997,-0.0361,...,-0.173,0.0336,-0.259,-0.16,-0.191,-0.182,-0.214,-0.0804,-0.19,0.0019
5,0.0329,-0.0331,0.159,0.0279,-0.0492,0.0725,-0.0281,0.127,0.042,-0.395,...,0.0051,-0.0118,-0.145,-0.0683,-0.0757,-0.0749,-0.213,-0.101,-0.0948,0.0056
6,0.0325,-0.0103,0.164,-0.0884,0.0631,-0.0556,0.0209,0.127,0.0115,0.0051,...,0.0301,0.027,0.0856,-0.0098,0.0457,0.0579,-0.026,0.178,0.232,0.0497


In [27]:
ge2_data = ge2_data.apply(pd.to_numeric, errors='coerce')

# Ensures all values are numeric
# If any hidden text exists → converted to NaN
# Important before further processing

In [28]:
print(ge2_data.dtypes.unique())
print(ge2_data.isnull().sum().sum())

[dtype('float64')]
342


In [29]:
ge2_data = ge2_data.astype(np.float32)

# Reduce memory usage (important for large dataset)
# Speeds up computation

In [30]:
print(ge2_data.dtypes.unique())

[dtype('float32')]


In [31]:
data_np2 = ge2_data.values

# WHY:
# NumPy is much faster for large computations
# Required for efficient mean imputation

In [32]:
print(type(data_np2))
print(data_np2.shape)

<class 'numpy.ndarray'>
(39280, 230)


In [33]:
col_mean2 = np.nanmean(data_np2, axis=0)

# Compute mean for each column (sample-wise)
# Needed to replace missing values
# np.nanmean ignores NaN values

In [34]:
print(col_mean2.shape)
print(np.isnan(col_mean2).sum())

(230,)
0


In [35]:
inds2 = np.where(np.isnan(data_np2))

# Find positions where values are missing (NaN)
# We will replace them using column mean

In [36]:
print(len(inds2[0]))

342


In [37]:
data_np2[inds2] = np.take(col_mean2, inds2[1])

# inds2[1] → column indices
# np.take(col_mean2, inds2[1]) → picks correct mean for each column
# Replaces each NaN with corresponding column mean

In [38]:
print(np.isnan(data_np2).sum())

0


In [39]:
ge2_data = pd.DataFrame(data_np2, columns=ge2_data.columns)

# Convert NumPy array back to pandas DataFrame
# Easier handling for next steps

In [40]:
print(type(ge2_data))
print(ge2_data.shape)

<class 'pandas.core.frame.DataFrame'>
(39280, 230)


In [41]:
ge2_data = pd.DataFrame(ge2_data.values.T, columns=gene_ids)

# Current: Genes × Samples → (39280, 230)
# Needed:  Samples × Genes → (230, 39280)
# ML models require rows = samples, columns = features
# Using NumPy transpose for speed

In [42]:
print(ge2_data.shape)
print(ge2_data.columns[:5])

(230, 39280)
Index(['10019475365', '10019481149', '10019495284', '10019687586',
       '10019713746'],
      dtype='object', name=0)


In [43]:
ge2_data.columns = ge2_data.columns.astype(str)

# Ensure all feature names are strings
# Required for sklearn compatibility

In [44]:
print(type(ge2_data.columns[0]))

<class 'str'>


In [45]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
ge2_data = scaler.fit_transform(ge2_data)

# Apply Z-score normalization
# Mean = 0, Std = 1
# Ensures all genes are on same scale
# Same method used in GSE33000 → consistency

In [46]:
print(type(ge2_data))
print(ge2_data.shape)
print(np.mean(ge2_data))
print(np.std(ge2_data))

<class 'numpy.ndarray'>
(230, 39280)
-6.35051e-10
1.0


In [47]:
print(np.isnan(ge2_data).sum())

# Ensure no NaN values remain after imputation + normalization

0


In [48]:
print(np.isnan(ge2_data).sum())

# Ensure no NaN values remain after imputation + normalization

0


In [49]:
print(np.isfinite(ge2_data).all())

# Checks for NaN, +inf, -inf
# Ensures data is completely valid

True


In [50]:
np.save("../data/processed/GSE44770_data.npy", ge2_data)
np.save("../data/processed/GSE44770_labels.npy", ge2_labels)

# Save processed dataset
# Avoid recomputation
# Ready for ML pipeline

In [51]:
# Dataset Integration (GSE33000 + GSE44770)

In [53]:
ge1_data = np.load("../data/processed/GSE33000_data.npy")
ge1_labels = np.load("../data/processed/GSE33000_labels.npy")

ge2_data = np.load("../data/processed/GSE44770_data.npy")
ge2_labels = np.load("../data/processed/GSE44770_labels.npy")

# WHY:
# Load preprocessed datasets for integration

In [54]:
print("GE1 shape:", ge1_data.shape)
print("GE2 shape:", ge2_data.shape)

print("GE1 labels:", len(ge1_labels))
print("GE2 labels:", len(ge2_labels))

GE1 shape: (467, 39280)
GE2 shape: (230, 39280)
GE1 labels: 467
GE2 labels: 230


In [55]:
ge1_df = pd.DataFrame(ge1_data)
ge2_df = pd.DataFrame(ge2_data)

# WHY:
# Convert numpy arrays to DataFrame
# So we can compare structure

In [56]:
print(ge1_df.shape[1], ge2_df.shape[1])

39280 39280


In [57]:
print((ge1_df.columns == ge2_df.columns).all())

True


In [58]:
combined_data = np.vstack((ge1_data, ge2_data))

# Stack datasets vertically
# Rows = samples → so we append samples

In [59]:
combined_labels = np.concatenate((ge1_labels, ge2_labels))

# Combine labels corresponding to samples

In [60]:
print("Combined data shape:", combined_data.shape)
print("Combined labels:", len(combined_labels))

Combined data shape: (697, 39280)
Combined labels: 697


In [61]:
unique, counts = np.unique(combined_labels, return_counts=True)

print(dict(zip(unique, counts)))

# Check how many samples in each class (0 and 1)
# Important for model performance

{np.int64(0): np.int64(257), np.int64(1): np.int64(440)}


In [62]:
np.save("../data/processed/combined_data.npy", combined_data)
np.save("../data/processed/combined_labels.npy", combined_labels)

# Save final dataset
# Avoid recomputation
# Ready for model training

In [63]:
import os
print(os.listdir("../data/processed"))

['.gitkeep', 'combined_data.npy', 'combined_labels.npy', 'GSE33000_data.npy', 'GSE33000_labels.npy', 'GSE44770_data.npy', 'GSE44770_labels.npy']
